# Day 18 — Zero-Shot Ag-Classification with CLIP

**Agri-AI Bootcamp** · Phase 5 of 5: *Transformers & Ag-Agents* · Days 17–20

> Load openai/clip-vit-base-patch32 and classify unlabelled crop images with custom text labels.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

CLIP trains an image encoder and a text encoder so matching pairs land close together. To classify, embed the image and each candidate sentence, then softmax the similarities. New classes need only new text, with no retraining.

### 🌾 Agriculture example

Candidate labels: “healthy wheat field”, “dry patch of land needing water”, “crop field damaged by pests”. The highest probability wins.

> **Watch out:** Wording matters; try several phrasings, and don’t treat the output as a diagnosis.

## 2. Setup

In [ ]:
!pip -q install transformers

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

## 3. Data

In [ ]:
# ---- Unlabelled field images -------------------------------------------------
# Upload your own:  from google.colab import files; files.upload()
# Here we synthesise three distinguishable "fields" so the notebook always runs.
rng = np.random.default_rng(0)
def field(kind, size=224):
    a = rng.uniform(0, .05, (size, size, 3)) + {
        "healthy": [.18, .55, .20], "dry": [.70, .62, .35], "pest": [.35, .45, .18]
    }[kind]
    if kind == "pest":
        for _ in range(60):
            x, y = rng.integers(0, size, 2); r = rng.integers(3, 9)
            yy, xx = np.ogrid[:size, :size]
            a[(xx-x)**2 + (yy-y)**2 <= r**2] = [.42, .30, .12]
    if kind == "dry":
        a += rng.uniform(-.06, .06, (size, size, 1))
    return Image.fromarray((a.clip(0,1)*255).astype(np.uint8))

images = {k: field(k) for k in ["healthy", "dry", "pest"]}
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, (k, im) in zip(ax, images.items()):
    a.imshow(im); a.set_title(f"(truth: {k})", fontsize=9); a.axis("off")
plt.show()

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
from transformers import CLIPModel, CLIPProcessor
n="openai/clip-vit-base-patch32"
m=CLIPModel.from_pretrained(n); pr=CLIPProcessor.from_pretrained(n)
labels=["healthy wheat field","dry patch of land needing water","crop field damaged by pests"]
x=pr(text=labels,images=img,return_tensors="pt",padding=True)
print(m(**x).logits_per_image.softmax(1))
```

## 5. Worked solution

In [ ]:
name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(name).to(dev).eval()
proc  = CLIPProcessor.from_pretrained(name)
print("params:", f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M",
      "| image tower:", model.config.vision_config.model_type)

LABELS = ["a healthy green wheat field",
          "a dry patch of land needing water",
          "a crop field damaged by pests"]

def zero_shot(img, labels):
    x = proc(text=labels, images=img, return_tensors="pt", padding=True).to(dev)
    with torch.no_grad():
        p = model(**x).logits_per_image.softmax(1)[0].cpu().numpy()
    return dict(zip(labels, p.round(3)))

print("\n--- zero-shot, no training at all ---")
for truth, img in images.items():
    scores = zero_shot(img, LABELS)
    best = max(scores, key=scores.get)
    print(f"\ntruth={truth}")
    for k, v in sorted(scores.items(), key=lambda kv: -kv[1]):
        print(f"   {v:.3f}  {k}")
    print(f"   -> picked: {best}")

# ---- New classes cost nothing but new sentences -----------------------------
NEW = LABELS + ["a flooded rice paddy", "bare soil before sowing", "a field of yellow flowers"]
print("\n--- six classes, still zero retraining ---")
s = zero_shot(images["dry"], NEW)
for k, v in sorted(s.items(), key=lambda kv: -kv[1])[:3]:
    print(f"   {v:.3f}  {k}")

# ---- Prompt wording changes the answer --------------------------------------
print("\n--- the same picture, three phrasings ---")
for variant in [["healthy crop", "unhealthy crop"],
                ["a photo of a healthy crop field", "a photo of a diseased crop field"],
                ["good", "bad"]]:
    s = zero_shot(images["pest"], variant)
    print(f"   {variant} -> {s}")
print("\nPrompt engineering IS part of the model here. Try several phrasings and keep"
      "\nthe wording fixed once you report numbers.")

# ---- The embedding space is shared ------------------------------------------
with torch.no_grad():
    it = model.get_image_features(**proc(images=list(images.values()),
                                         return_tensors="pt").to(dev))
    tt = model.get_text_features(**proc(text=LABELS, return_tensors="pt",
                                        padding=True).to(dev))
it = it / it.norm(dim=-1, keepdim=True); tt = tt / tt.norm(dim=-1, keepdim=True)
sim = (it @ tt.T).cpu().numpy()
plt.imshow(sim, cmap="viridis"); plt.colorbar(label="cosine similarity")
plt.xticks(range(3), ["healthy", "dry", "pest"]); plt.yticks(range(3), list(images))
plt.xlabel("text"); plt.ylabel("image"); plt.title("image-text similarity"); plt.show()
print("\nNOT a diagnosis: CLIP matches wording to appearance. An agronomist confirms.")

## 6. Your turn

1. Write 5 prompt variants for one disease and report the spread in probability. That spread is your uncertainty.
2. Add a 'none of the above' label and see whether it absorbs the uncertain images.
3. Run CLIP on real PlantVillage images and compare with your Day 11 fine-tuned model.

In [ ]:
# TODO 1: prompt sensitivity
variants = ["a diseased leaf", "a photo of a diseased plant leaf",
            "leaf with brown spots", "sick plant", "a close-up photo of leaf blight"]
for v in variants:
    s = zero_shot(images["pest"], [v, "a healthy plant leaf"])
    print(f"{s[v]:.3f}   {v}")

# TODO 2: add "none of the above"
# TODO 3: compare against the Day 11 supervised model on the same images

## 7. Dataset and references

**Dataset:** [Kaggle: PlantVillage leaf images](https://www.kaggle.com/datasets/emmarex/plantdisease)

**References**

- [OpenAI: CLIP](https://openai.com/index/clip/)
- [CLIP model card](https://huggingface.co/openai/clip-vit-base-patch32)

**Course material:** OpenAI Blog: CLIP

---
**Next:** Day 19 — Vision-Language Modeling for Diagnosis